<a href="https://colab.research.google.com/github/DerWeiseTeufel/MVM_2025/blob/main/Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("all_models.csv")
df.head()

,model,prompt,image_path,output,elapsed_time_s
0,gemini-2.0-flash,You are an image–analysis model. You will be s...,/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...,"```json\n{\n ""terrain"": ""urban"",\n ""objects""...",1.695201
1,gemini-2.0-flash,You are an image–analysis model. You will be s...,/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...,"```json\n{\n ""terrain"": ""field"",\n ""objects""...",1.035956
2,gemini-2.0-flash,You are an image–analysis model. You will be s...,/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...,"```json\n{\n ""terrain"": ""field"",\n ""objects""...",1.321373
3,gemini-2.0-flash,You are an image–analysis model. You will be s...,/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...,"```json\n{\n ""terrain"": ""urban"",\n ""objects""...",1.254391
4,gemini-2.0-flash,You are an image–analysis model. You will be s...,/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...,"```json\n{\n ""terrain"": ""urban"",\n ""objects""...",1.352515


In [ ]:
df_true = pd.read_excel("benchmark.xlsx")
df_true.head()

,terrain,objects,chatGPTAnswer,image_path
0,field,natural_objects military_vehicles,"{\n ""terrain"": ""field"",\n ""objects"": [""m...",/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...
1,field,natural_objects military_vehicles,"{\n ""terrain"": ""field"",\n ""objects"": [""m...",/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...
2,forest,natural_objects vehicles,"{\n ""terrain"": ""forest"",\n ""objects"": [""...",/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...
3,field,military_vehicles natural_objects,"{\n ""terrain"": ""field"",\n ""objects"": [""v...",/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...
4,field,natural_objects vehicles,"{\n ""terrain"": ""field"",\n ""objects"": [""r...",/content/drive/MyDrive/AIWeekKrasnoyarsk/Image...


In [ ]:
import json
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def normalize_category(cat: str):
    """
    Convert model-style categories like 'military vehicles'
    into GT-style 'military_vehicles'.
    """
    if not isinstance(cat, str):
        return ""
    return cat.strip().lower().replace(" ", "_")


def normalize_list(categories):
    """
    Normalize a list of category strings.
    """
    return [normalize_category(c) for c in categories if isinstance(c, str)]


def parse_gt_objects(obj_str: str):
    if not isinstance(obj_str, str):
        return []
    return [normalize_category(x) for x in obj_str.split()]


def load_ground_truth(df_gt):
    gt = {}
    for _, row in df_gt.iterrows():
        gt[row.image_path] = {
            "terrain": row.terrain,
            "objects": parse_gt_objects(row.objects),
            "description": row.chatGPTAnswer,
        }
    return gt



def parse_output(output_str):
    try:
        data = json.loads(output_str)
        return {
            "terrain": data.get("terrain"),
            "objects": normalize_list(data.get("objects", [])),
            "description": data.get("description", "")
        }
    except Exception:
        return {"terrain": None, "objects": [], "description": ""}



def terrain_accuracy(df, GT):
    correct = 0
    for _, row in df.iterrows():
        if parse_output(row.output)["terrain"] == GT[row.image_path]["terrain"]:
            correct += 1
    return correct / len(df) if len(df) > 0 else 0


def object_accuracy(df, GT):
    correct = 0
    all_obj = 0
    for _, row in df.iterrows():
        gt_set = set(GT[row.image_path]["objects"])
        all_obj+=len(gt_set)
        pred_str = " ".join(parse_output(row.output)["objects"])
        all_obj+=len(set(parse_output(row.output)["objects"]))

        count_cur=0
        for obj in gt_set:
          if pred_str.count(obj)>0:
            correct+=1
            count_cur+=1
        all_obj-=count_cur

    return correct / all_obj if all_obj > 0 else 0


def bleu_score(df, GT):
    chencherry = SmoothingFunction()
    scores = []
    for _, row in df.iterrows():
        pred_desc = parse_output(row.output)["description"]
        gt_desc = GT[row.image_path]["description"]

        if not pred_desc or not gt_desc:
            scores.append(0)
            continue

        score = sentence_bleu(
            [gt_desc.split()],
            pred_desc.split(),
            smoothing_function=chencherry.method1
        )
        scores.append(score)

    return sum(scores) / len(scores) if scores else 0
def time_score(df):
  return float(df['elapsed_time_s'].mean())



def evaluate_model(df_results, df_gt, model_name):
    GT = load_ground_truth(df_gt)
    df = df_results[df_results['model'] == model_name]

    return {
        "terr_acc": terrain_accuracy(df, GT),
        "object_acc": object_accuracy(df, GT),
        "bleu_score": bleu_score(df, GT),
        "average_time_s": time_score(df)
    }



model_names = df['model'].unique()
results_df = pd.read_csv("all_models.csv")
results_df = results_df.dropna()
results_df['output'] = results_df['output'].apply(lambda x: x.replace("```", "").replace("json", ""))
gt_df = pd.read_excel("benchmark.xlsx")
metrics_bench = {}
metrics_bench[model_names[0]] = evaluate_model(results_df, gt_df, model_names[0])
metrics_bench[model_names[1]] = evaluate_model(results_df, gt_df, model_names[1])
metrics_bench[model_names[2]] = evaluate_model(results_df, gt_df, model_names[2])


In [ ]:
metrics_bench

{'gemini-2.0-flash': {'terr_acc': 0.7,
  'object_acc': 0.527027027027027,
  'bleu_score': 0.02161075617538816,
  'average_time_s': 1.4192834615707397},
 'Qwen/Qwen2.5-VL-7B-Instruct:hyperbolic': {'terr_acc': 0.65,
  'object_acc': 0.3188405797101449,
  'bleu_score': 0.028510548297917836,
  'average_time_s': 1.1902342200279237},
 'meta-llama/Llama-4-Scout-17B-16E-Instruct:groq': {'terr_acc': 0.7,
  'object_acc': 0.49295774647887325,
  'bleu_score': 0.035021574252476574,
  'average_time_s': 0.45007368326187136}}

In [ ]:
import plotly.graph_objects as go

model_scores = metrics_bench
models = list(model_scores.keys())
metrics = ["terr_acc", "object_acc", "bleu_score"]

traces = []
for metric in metrics:
    traces.append(
        go.Bar(
            name=metric,
            x=models,
            y=[model_scores[m][metric] for m in models]
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    barmode='group',
    title="Сравнение моделей",
    yaxis=dict(title="Скор"),
    xaxis=dict(title="Модель")
)
fig.show()
